# Model 3 Stage 2: Load Gated Checkpoint, Prune, Finetune, Evaluate

This notebook starts from an already-trained gated Model 3 checkpoint (`best_gated_cross_attention_model.pth`) and runs the remaining steps: gate analysis, pruning sweep, finetuning, and final evaluation.


In [ ]:
import json
import math
import os
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

BATCH_SIZE = 32
EPOCHS = 50
FINETUNE_EPOCHS = 10
LEARNING_RATE = 1e-4
MAX_LENGTH = 128
RANDOM_SEED = 42
DATA_FILE = 'Cleaned_LOOPerSet_GEM.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FEATURE_COLS = [
    'T1', 'T2', 'Unroll', 'locality', 'working_set_size',
    'arithmetic_intensity', 'T1_T2_inter', 'log_T1', 'log_T2', 'T1_div_T2'
]
GATE_LAMBDA = 1e-3
HC_BETA = 2.0 / 3.0
HC_GAMMA = -0.1
HC_ZETA = 1.1
GATE_THRESHOLD = 0.5
PRUNING_RATIOS = [0.10, 0.20, 0.30, 0.40, 0.50]
PRETRAINED_MODEL2_PATH = 'best_cross_attention_model.pth'
GATED_MODEL_PATH = 'best_gated_cross_attention_model.pth'
BEST_MODEL_PATH = 'best_cross_pruned_model.pth'
PRUNING_INFO_PATH = 'best_cross_pruned_info.pth'
METRICS_PATH = 'model3_metrics.json'

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f'[INFO] Using device: {DEVICE}')

In [ ]:
def compute_regression_metrics(actual_ms, predicted_ms):
    actual_ms = np.asarray(actual_ms, dtype=np.float64)
    predicted_ms = np.asarray(predicted_ms, dtype=np.float64)
    residuals = predicted_ms - actual_ms
    absolute_errors = np.abs(residuals)
    mae = float(np.mean(absolute_errors))
    rmse = float(np.sqrt(np.mean(np.square(residuals))))
    median_ae = float(np.median(absolute_errors))
    safe_actual = np.clip(np.abs(actual_ms), 1e-6, None)
    mape = float(np.mean(absolute_errors / safe_actual) * 100.0)
    ss_res = float(np.sum(np.square(residuals)))
    ss_tot = float(np.sum(np.square(actual_ms - actual_ms.mean())))
    r2 = float(1.0 - (ss_res / ss_tot)) if ss_tot > 0 else float('nan')
    return {
        'mae_ms': mae,
        'rmse_ms': rmse,
        'median_ae_ms': median_ae,
        'mape_percent': mape,
        'r2': r2,
        'residuals': residuals,
        'absolute_errors': absolute_errors,
    }


def evaluate_model(model, data_loader, criterion):
    model.eval()
    total_loss = 0.0
    predictions_log = []
    labels_log = []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            schedule_features = batch['schedule_features'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            outputs = model(input_ids, attention_mask, schedule_features).squeeze(-1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            predictions_log.append(outputs.cpu().numpy())
            labels_log.append(labels.cpu().numpy())

    predictions_log = np.concatenate(predictions_log)
    labels_log = np.concatenate(labels_log)
    predicted_ms = np.expm1(predictions_log)
    actual_ms = np.expm1(labels_log)
    metrics = compute_regression_metrics(actual_ms, predicted_ms)
    metrics['avg_loss'] = total_loss / max(len(data_loader), 1)
    metrics['predicted_ms'] = predicted_ms
    metrics['actual_ms'] = actual_ms
    return metrics

In [ ]:
class CompilerDataset(Dataset):
    def __init__(self, dataframe, tokenizer, feature_means, feature_stds, max_length=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.feature_means = np.asarray(feature_means, dtype=np.float32)
        self.feature_stds = np.asarray(feature_stds, dtype=np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoding = self.tokenizer(
            row['Code_String'],
            return_tensors='pt',
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
        )
        raw_features = np.array([
            float(row['T1']),
            float(row['T2']),
            float(row['Unroll']),
            float(row['locality']),
            float(row['working_set_size']),
            float(row['arithmetic_intensity']),
            float(row['T1_T2_inter']),
            float(row['log_T1']),
            float(row['log_T2']),
            float(row['T1_div_T2']),
        ], dtype=np.float32)
        scaled_features = (raw_features - self.feature_means) / self.feature_stds
        label = float(row['log_execution_time'])
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'schedule_features': torch.tensor(scaled_features, dtype=torch.float32),
            'labels': torch.tensor(label, dtype=torch.float32),
        }

In [ ]:
class HardConcreteHeadGate(nn.Module):
    def __init__(self, num_heads, beta=HC_BETA, gamma=HC_GAMMA, zeta=HC_ZETA):
        super().__init__()
        self.log_alpha = nn.Parameter(torch.zeros(num_heads))
        self.beta = beta
        self.gamma = gamma
        self.zeta = zeta

    def _stretch(self, s):
        return s * (self.zeta - self.gamma) + self.gamma

    def sample_gates(self):
        if self.training:
            u = torch.rand_like(self.log_alpha).clamp(1e-6, 1 - 1e-6)
            s = torch.sigmoid((torch.log(u) - torch.log(1 - u) + self.log_alpha) / self.beta)
        else:
            s = torch.sigmoid(self.log_alpha)
        z = self._stretch(s)
        return z.clamp(0.0, 1.0)

    def gate_probabilities(self):
        offset = self.beta * math.log(-self.gamma / self.zeta)
        return torch.sigmoid(self.log_alpha - offset)

    def expected_l0(self):
        return self.gate_probabilities().sum()

    def expected_gates(self):
        z = self._stretch(torch.sigmoid(self.log_alpha))
        return z.clamp(0.0, 1.0)


class GatedSelfAttention(nn.Module):
    def __init__(self, original_self_attention):
        super().__init__()
        self.query = original_self_attention.query
        self.key = original_self_attention.key
        self.value = original_self_attention.value
        self.dropout = original_self_attention.dropout
        self.position_embedding_type = getattr(original_self_attention, 'position_embedding_type', 'absolute')
        self.is_decoder = getattr(original_self_attention, 'is_decoder', False)
        self.num_attention_heads = original_self_attention.num_attention_heads
        self.attention_head_size = original_self_attention.attention_head_size
        self.all_head_size = original_self_attention.all_head_size
        self.gate = HardConcreteHeadGate(self.num_attention_heads)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        past_key_value=None,
        output_attentions=False,
        cache_position=None,
        **kwargs,
    ):
        mixed_query_layer = self.query(hidden_states)
        is_cross_attention = encoder_hidden_states is not None

        if is_cross_attention:
            key_layer = self.transpose_for_scores(self.key(encoder_hidden_states))
            value_layer = self.transpose_for_scores(self.value(encoder_hidden_states))
            attention_mask = encoder_attention_mask
        else:
            key_layer = self.transpose_for_scores(self.key(hidden_states))
            value_layer = self.transpose_for_scores(self.value(hidden_states))

        query_layer = self.transpose_for_scores(mixed_query_layer)
        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)

        if attention_mask is not None:
            attention_scores = attention_scores + attention_mask

        attention_probs = torch.softmax(attention_scores, dim=-1)
        attention_probs = self.dropout(attention_probs)

        if head_mask is not None:
            attention_probs = attention_probs * head_mask

        context_layer = torch.matmul(attention_probs, value_layer)
        gates = self.gate.sample_gates().view(1, self.num_attention_heads, 1, 1)
        context_layer = context_layer * gates

        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(new_context_layer_shape)

        attn_weights = attention_probs if output_attentions else None
        outputs = (context_layer, attn_weights)
        if self.is_decoder:
            outputs = outputs + (past_key_value,)
        return outputs


def inject_gates_into_codebert(codebert_model):
    for layer in codebert_model.encoder.layer:
        if not isinstance(layer.attention.self, GatedSelfAttention):
            layer.attention.self = GatedSelfAttention(layer.attention.self)
    return codebert_model


def collect_gate_modules(codebert_model):
    modules = []
    for layer_idx, layer in enumerate(codebert_model.encoder.layer):
        if isinstance(layer.attention.self, GatedSelfAttention):
            modules.append((layer_idx, layer.attention.self.gate))
    return modules


def get_gate_dataframe(codebert_model):
    records = []
    for layer_idx, gate in collect_gate_modules(codebert_model):
        probs = gate.gate_probabilities().detach().cpu().numpy()
        expected = gate.expected_gates().detach().cpu().numpy()
        for head_idx, (prob, exp_gate) in enumerate(zip(probs, expected)):
            records.append({
                'layer': layer_idx,
                'head': head_idx,
                'gate_probability': float(prob),
                'expected_gate': float(exp_gate),
            })
    return pd.DataFrame(records)


def compute_gate_sparsity_loss(codebert_model):
    losses = []
    total_active = []
    for _, gate in collect_gate_modules(codebert_model):
        losses.append(gate.expected_l0())
        total_active.append(gate.expected_gates().sum())
    if not losses:
        zero = torch.tensor(0.0, device=DEVICE)
        return zero, zero
    return torch.stack(losses).sum(), torch.stack(total_active).sum()

In [ ]:
class BaseCrossAttentionCostModel(nn.Module):
    def __init__(self, schedule_feature_size=10, attention_dim=128):
        super().__init__()
        self.codebert = AutoModel.from_pretrained('microsoft/codebert-base')
        self.schedule_mlp = nn.Sequential(
            nn.Linear(schedule_feature_size, 32),
            nn.GELU(),
            nn.Linear(32, 32),
            nn.GELU(),
        )
        self.query_proj = nn.Linear(32, attention_dim)
        self.key_proj = nn.Linear(768, attention_dim)
        self.value_proj = nn.Linear(768, attention_dim)
        self.regressor = nn.Sequential(
            nn.Linear(attention_dim + 32, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def encode_and_predict(self, input_ids, attention_mask, schedule_features, return_attention=False):
        code_outputs = self.codebert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = code_outputs.last_hidden_state
        schedule_embedding = self.schedule_mlp(schedule_features)
        query = self.query_proj(schedule_embedding).unsqueeze(1)
        keys = self.key_proj(token_embeddings)
        values = self.value_proj(token_embeddings)
        scores = torch.matmul(query, keys.transpose(1, 2)) / math.sqrt(keys.size(-1))
        mask = attention_mask.unsqueeze(1) == 0
        scores = scores.masked_fill(mask, float('-inf'))
        attention_weights = torch.softmax(scores, dim=-1)
        attended_code = torch.matmul(attention_weights, values).squeeze(1)
        fused = torch.cat((attended_code, schedule_embedding), dim=1)
        prediction = self.regressor(fused)
        if return_attention:
            return prediction, attention_weights.squeeze(1)
        return prediction


class GatedCrossAttentionCostModel(BaseCrossAttentionCostModel):
    def __init__(self, schedule_feature_size=10, attention_dim=128):
        super().__init__(schedule_feature_size=schedule_feature_size, attention_dim=attention_dim)
        inject_gates_into_codebert(self.codebert)

    def forward(self, input_ids, attention_mask, schedule_features, return_attention=False):
        return self.encode_and_predict(input_ids, attention_mask, schedule_features, return_attention=return_attention)


class PrunedCrossAttentionCostModel(BaseCrossAttentionCostModel):
    def __init__(self, schedule_feature_size=10, attention_dim=128):
        super().__init__(schedule_feature_size=schedule_feature_size, attention_dim=attention_dim)

    def forward(self, input_ids, attention_mask, schedule_features, return_attention=False):
        return self.encode_and_predict(input_ids, attention_mask, schedule_features, return_attention=return_attention)

In [ ]:
print('[INFO] Loading tokenizer and dataset...')
tokenizer = AutoTokenizer.from_pretrained('microsoft/codebert-base')
full_df = pd.read_csv(DATA_FILE)
num_rows = len(full_df)

generator = torch.Generator().manual_seed(RANDOM_SEED)
permutation = torch.randperm(num_rows, generator=generator).tolist()
train_size = int(0.8 * num_rows)
remaining_size = num_rows - train_size
val_size = remaining_size // 2

a_idx = permutation[:train_size]
b_idx = permutation[train_size:train_size + val_size]
c_idx = permutation[train_size + val_size:]

train_df = full_df.iloc[a_idx].reset_index(drop=True)
val_df = full_df.iloc[b_idx].reset_index(drop=True)
test_df = full_df.iloc[c_idx].reset_index(drop=True)

feature_means = train_df[FEATURE_COLS].mean().to_numpy(dtype=np.float32)
feature_stds = train_df[FEATURE_COLS].std().replace(0, 1.0).to_numpy(dtype=np.float32)
feature_stats = {
    'means': feature_means.tolist(),
    'stds': feature_stds.tolist(),
    'feature_cols': FEATURE_COLS,
}

train_dataset = CompilerDataset(train_df, tokenizer, feature_means, feature_stds, max_length=MAX_LENGTH)
val_dataset = CompilerDataset(val_df, tokenizer, feature_means, feature_stds, max_length=MAX_LENGTH)
test_dataset = CompilerDataset(test_df, tokenizer, feature_means, feature_stds, max_length=MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'[INFO] Split sizes -> train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}')
print('[INFO] First three feature means:', feature_means[:3])
print('[INFO] First three feature stds:', feature_stds[:3])

In [ ]:
model = GatedCrossAttentionCostModel(schedule_feature_size=len(FEATURE_COLS)).to(DEVICE)
criterion = nn.MSELoss()

print('[INFO] Stage 2 notebook ready.')
print('[INFO] Expected input checkpoint:', GATED_MODEL_PATH)
print('[INFO] This notebook skips gated training and starts from the saved best checkpoint.')

In [ ]:
model = GatedCrossAttentionCostModel(schedule_feature_size=len(FEATURE_COLS)).to(DEVICE)
criterion = nn.MSELoss()

print('[INFO] Stage 2 notebook ready.')
print('[INFO] Expected input checkpoint:', GATED_MODEL_PATH)
print('[INFO] This notebook skips gated training and starts from the saved best checkpoint.')

In [ ]:
if 'gate_df' not in globals():
    if not os.path.exists(GATED_MODEL_PATH):
        raise FileNotFoundError(
            f"Missing gated checkpoint: {GATED_MODEL_PATH}. Copy your saved best gated model into this notebook directory first."
        )
    checkpoint = torch.load(GATED_MODEL_PATH, map_location=DEVICE)
    best_gated_model = GatedCrossAttentionCostModel(schedule_feature_size=len(FEATURE_COLS)).to(DEVICE)
    best_gated_model.load_state_dict(checkpoint['model_state_dict'])
    best_gated_model.eval()
    gate_df = get_gate_dataframe(best_gated_model.codebert).sort_values('expected_gate', ascending=True).reset_index(drop=True)
    print('[INFO] gate_df was missing, so it was rebuilt from the saved gated checkpoint.')

def build_pruning_plan_from_ratio(gate_df, pruning_ratio):
    num_total_heads = len(gate_df)
    num_to_prune = max(1, int(num_total_heads * pruning_ratio))
    weakest = gate_df.nsmallest(num_to_prune, 'expected_gate')
    pruning_plan = defaultdict(set)
    for _, row in weakest.iterrows():
        pruning_plan[int(row['layer'])].add(int(row['head']))
    threshold = float(weakest['expected_gate'].max())
    return dict(pruning_plan), num_to_prune, threshold


def apply_gate_mask(model, pruning_plan, off_value=-20.0):
    for layer_idx, head_indices in pruning_plan.items():
        gate = model.codebert.encoder.layer[int(layer_idx)].attention.self.gate
        with torch.no_grad():
            gate.log_alpha[list(head_indices)] = off_value
    return model


def freeze_gate_parameters(model):
    for _, gate in collect_gate_modules(model.codebert):
        for param in gate.parameters():
            param.requires_grad = False
    return model


def count_functionally_pruned_heads(model, threshold=0.01):
    gate_df_local = get_gate_dataframe(model.codebert)
    return int((gate_df_local['expected_gate'] <= threshold).sum())


def build_pruned_model(pruning_plan):
    pruned_model = GatedCrossAttentionCostModel(schedule_feature_size=len(FEATURE_COLS)).to(DEVICE)
    pruned_model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    pruned_model = apply_gate_mask(pruned_model, pruning_plan)
    pruned_model = freeze_gate_parameters(pruned_model)
    for param in pruned_model.codebert.parameters():
        param.requires_grad = False
    return pruned_model, count_functionally_pruned_heads(pruned_model)


def finetune_pruned_model(pruning_plan):
    pruned_model, pruned_head_count = build_pruned_model(pruning_plan)
    optimizer = AdamW(filter(lambda p: p.requires_grad, pruned_model.parameters()), lr=LEARNING_RATE)
    history = {'train_loss': [], 'val_loss': [], 'val_mae': []}
    best_state = None
    best_val_mae = float('inf')
    best_val_loss = float('inf')

    for epoch in range(FINETUNE_EPOCHS):
        pruned_model.train()
        pruned_model.codebert.eval()
        running_loss = 0.0

        for batch in train_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            schedule_features = batch['schedule_features'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            optimizer.zero_grad()
            outputs = pruned_model(input_ids, attention_mask, schedule_features).squeeze(-1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / max(len(train_loader), 1)
        val_metrics = evaluate_model(pruned_model, val_loader, criterion)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_metrics['avg_loss'])
        history['val_mae'].append(val_metrics['mae_ms'])

        print(
            f"  [Finetune] Epoch {epoch + 1}/{FINETUNE_EPOCHS} | "
            f"train_loss={avg_train_loss:.4f} | val_loss={val_metrics['avg_loss']:.4f} | val_mae={val_metrics['mae_ms']:.2f} ms"
        )

        if val_metrics['mae_ms'] < best_val_mae:
            best_val_mae = val_metrics['mae_ms']
            best_val_loss = val_metrics['avg_loss']
            best_state = {k: v.detach().cpu().clone() for k, v in pruned_model.state_dict().items()}

    pruned_model.load_state_dict(best_state)
    return pruned_model, best_val_loss, best_val_mae, history, pruned_head_count


pruning_results = []
best_pruned_mae = float('inf')
best_retrain_history = None

for pruning_ratio in PRUNING_RATIOS:
    pruning_plan, num_pruned_heads, threshold = build_pruning_plan_from_ratio(gate_df, pruning_ratio)
    print(f'[INFO] Testing pruning ratio {pruning_ratio:.0%} ({num_pruned_heads} target heads)...')
    pruned_model, pruned_val_loss, pruned_val_mae, retrain_history, pruned_head_count = finetune_pruned_model(pruning_plan)
    pruning_results.append({
        'pruning_ratio': pruning_ratio,
        'num_pruned_heads': pruned_head_count,
        'gate_threshold': threshold,
        'val_loss': pruned_val_loss,
        'val_mae': pruned_val_mae,
    })

    if pruned_val_mae < best_pruned_mae:
        best_pruned_mae = pruned_val_mae
        best_retrain_history = retrain_history
        torch.save(pruned_model.state_dict(), BEST_MODEL_PATH)
        torch.save({
            'pruning_plan': pruning_plan,
            'pruning_ratio': pruning_ratio,
            'num_pruned_heads': pruned_head_count,
            'gate_threshold': threshold,
            'feature_stats': feature_stats,
            'pruning_mode': 'functional_gate_masking'
        }, PRUNING_INFO_PATH)
        print('[INFO] Saved new best functionally pruned checkpoint.')

pruning_results_df = pd.DataFrame(pruning_results)
pruning_results_df

In [ ]:
pruning_info = torch.load(PRUNING_INFO_PATH, map_location='cpu')
plt.figure(figsize=(8, 4))
plt.plot(pruning_results_df['pruning_ratio'] * 100, pruning_results_df['val_mae'], marker='o')
plt.title('Accuracy vs Pruning Ratio')
plt.xlabel('Pruning ratio (%)')
plt.ylabel('Validation MAE (ms)')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

if best_retrain_history is not None:
    epochs = range(1, FINETUNE_EPOCHS + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, best_retrain_history['train_loss'], marker='o', label='Train Loss')
    axes[0].plot(epochs, best_retrain_history['val_loss'], marker='s', label='Val Loss')
    axes[0].set_title('Best Pruned Model Finetuning Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.4)

    axes[1].plot(epochs, best_retrain_history['val_mae'], marker='o', color='darkorange')
    axes[1].set_title('Best Pruned Model Validation MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE (ms)')
    axes[1].grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

In [ ]:
pruning_info = torch.load(PRUNING_INFO_PATH, map_location='cpu')
final_model = GatedCrossAttentionCostModel(schedule_feature_size=len(FEATURE_COLS)).to(DEVICE)
final_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE), strict=False)
final_model.eval()

test_metrics = evaluate_model(final_model, test_loader, criterion)
metrics_to_save = {
    'mae_ms': test_metrics['mae_ms'],
    'rmse_ms': test_metrics['rmse_ms'],
    'median_ae_ms': test_metrics['median_ae_ms'],
    'mape_percent': test_metrics['mape_percent'],
    'r2': test_metrics['r2'],
    'pruning_ratio': pruning_info['pruning_ratio'],
    'num_pruned_heads': pruning_info['num_pruned_heads'],
    'pruning_mode': pruning_info.get('pruning_mode', 'functional_gate_masking'),
}
with open(METRICS_PATH, 'w', encoding='utf-8') as fp:
    json.dump(metrics_to_save, fp, indent=2)

print('Test metrics:')
for key, value in metrics_to_save.items():
    if isinstance(value, float):
        print(f'  {key}: {value:.4f}')
    else:
        print(f'  {key}: {value}')

In [ ]:
actual_ms = test_metrics['actual_ms']
predicted_ms = test_metrics['predicted_ms']
residuals = test_metrics['residuals']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(actual_ms, predicted_ms, alpha=0.35)
min_val = max(min(actual_ms.min(), predicted_ms.min()), 1e-6)
max_val = max(actual_ms.max(), predicted_ms.max())
axes[0].plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_title('Predicted vs Actual')
axes[0].set_xlabel('Actual (ms)')
axes[0].set_ylabel('Predicted (ms)')

axes[1].scatter(actual_ms, residuals, alpha=0.35, color='darkorange')
axes[1].axhline(0.0, color='black', linestyle='--')
axes[1].set_xscale('log')
axes[1].set_title('Residuals vs Actual')
axes[1].set_xlabel('Actual (ms)')
axes[1].set_ylabel('Predicted - Actual (ms)')

axes[2].hist(residuals, bins=60, density=True, alpha=0.75, color='seagreen')
mu = residuals.mean()
sigma = residuals.std() if residuals.std() > 0 else 1.0
x = np.linspace(residuals.min(), residuals.max(), 400)
gaussian = (1.0 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
axes[2].plot(x, gaussian, color='black', linewidth=2)
axes[2].set_title('Error Distribution')
axes[2].set_xlabel('Residual (ms)')
axes[2].set_ylabel('Density')

plt.tight_layout()
plt.show()